## 2) Chat Models (OpenAI)

Think of Chat Models as different brands of TVs (Sony, Samsung, LG). Each one works differently on the inside, but LangChain gives you a **"Universal Remote"** to control them all.
This "Universal Remote" is called the **Runnable Interface**. It means you can press the "Play" button (`invoke`) on *any* model, and it will work exactly the same way. You don't need to learn a new remote control just because you bought a new TV.

### 🧠 Interface Concepts
all Chat Models implement the **Runnable Interface**. This guarantees they all speak the same language:
1.  **Standard Inputs:** They all accept a list of `Messages` (System, Human, AI).
2.  **Standard Outputs:** They all return an `AIMessage`.
3.  **Standard Methods:**
    * `.invoke()`: Send a message and wait for the full answer.
    * `.stream()`: Get the answer word-by-word (like a typewriter).
    * `.batch()`: Send 50 questions at once and get 50 answers back.

###  Code Context
In the code below:
* `init_chat_model`: A universal function to load a model.
* `model="gpt-4o-mini"`: We are selecting a specific, fast model from OpenAI.
* `temperature=0`: We set this to 0 to make the AI **factual and consistent**. Higher numbers (up to 1) make it more creative and random.

- Supports over 100 model providers : https://docs.langchain.com/oss/python/integrations/chat
- https://docs.langchain.com/oss/python/langchain/models

![Diagram](./images/02_img.png)

In [1]:
import os
import sys
from pathlib import Path

print(sys.executable)
project_root = Path.cwd()
while project_root != project_root.parent and not (
    project_root / "utils" / "env_utils.py"
).is_file():
    project_root = project_root.parent

if not (project_root / "utils" / "env_utils.py").is_file():
    raise FileNotFoundError("Could not locate utils/env_utils.py from the notebook directory.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from utils.env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week1-langchain/.venv/bin/python
Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.

Python 3.13.5 satisfies requires-python: >=3.11, <3.14
package                | required | installed | status | path                                                                            
---------------------- | -------- | --------- | ------ | --------------------------------------------------------------------------------
jupyter                | >=1.0.0  | 1.1.1     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packages
langchain              | >=1.3.10 | 1.4.5     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packages
langchain-anthropic    | >=1.4.6  | 1.7.6     | ✅ OK   | …ai-certification-gen-academy/week1-langchain/.venv/lib/python3.13/site-packag

In [3]:
from langchain_prompts import DATE_ASSISTANT_SYSTEM_PROMPT
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", DATE_ASSISTANT_SYSTEM_PROMPT),
    ("human", "{user_query}")
])

example_user_query = """
What is tomorrow's date?
"""

messages = prompt.format_messages(user_query = example_user_query, 
                                    todays_date = datetime.now().strftime("%Y-%m-%d"))  # Shows message objects ready for a chat model

messages


# Other Message Objects
    # System messages
    # Human messages
    # AIMessage
    # ToolMessage
    

[SystemMessage(content="\n\nYou are a helpful and accurate date assistant. \nYou are given a user's request, and today's date. Your job is to accurately infer the date in YYYY-MM-DD format from the user's request.\n\ntoday's date : 2026-10-13\n\nchat_history:\n\n", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="\nWhat is tomorrow's date?\n", additional_kwargs={}, response_metadata={})]

In [9]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="gpt-4o-mini", temperature=0)

## 3) LLM Invocation  

* **Invoke:** This is like sending a text message and waiting for the reply. You don't see anything until the whole message arrives.
* **Stream:** This is like watching someone type in real-time. You see the words appear one by one. This feels much faster to a user.

###  Code Context
In the code below:
* `llm.invoke(messages)`: Sends our formatted prompt to the AI and waits for the full `AIMessage` response.
* `llm.stream(messages)`: Returns a generator that prints tokens as they arrive.

📚 **[Docs](https://docs.langchain.com/oss/python/langchain/models#invocation)**

![Diagram](./images/03_img.png)

In [11]:
# Invoke:
response = llm.invoke(messages)
response

AIMessage(content="Tomorrow's date is 2026-10-14.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 73, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_bb65f3153e', 'id': 'chatcmpl-EYbnTVSXL8xplpvVyGytf3OdKo8Xp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13695-d22f-7e52-995e-1b5335c892a9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 12, 'total_tokens': 85, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 3.1) Streaming (Real-time Output)

Imagine you hired a ghostwriter to write a story.
* **Invoke (Standard):** The writer goes into a private room, locks the door, writes the entire story, and comes out 20 minutes later to hand you the finished manuscript. You sit in silence waiting the whole time.
* **Stream (Real-time):** The writer sits right in front of you and types. You see every letter appear on the screen the moment they press the key. The story takes the same amount of time to finish, but it **feels instant** because you are engaged immediately.

### Technical Concept
LLMs don't "think" of a whole sentence at once; they generate text **one token (word part) at a time**.
* **The Bottleneck:** By default (`.invoke()`), the program buffers (holds) all these tokens in memory until the model is completely finished. This creates **Latency** (the delay between asking and seeing the answer).
* **The Solution:** The `.stream()` method bypasses this buffer. It creates a Python **Generator** that yields each token (`AIMessageChunk`) the exact millisecond it is created. This drastically lowers the "Time to First Token."

### Code Context
In the code below:
* `llm.stream(messages)`: This replaces `.invoke()`. It returns an iterable stream, not a final string.
* `print(..., end="", flush=True)`: We force Python to print *immediately* without moving to a new line, creating that smooth "typewriter" effect.

📚 **[Docs: Streaming](https://docs.langchain.com/oss/python/langchain/streaming)**


In [13]:
# Streaming
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True )

Tomorrow's date is 2026-10-14.